# ChuckleNet Phase A + Prosody: Streaming + Checkpoints + Resume

## Fixes from v1:
- ✅ **Streaming audio** — no RAM cache, loads on-demand
- ✅ **Checkpoint every epoch** — saves to Drive after each epoch
- ✅ **Resume from checkpoint** — run SETUP + RESUME cell to continue
- ✅ **Fixed `epoch` scope bug** — now passed as parameter
- ✅ **Fixed batch count** — `math.ceil` instead of float division
- ✅ **NaN/Inf loss check** — skips batch if loss diverges
- ✅ **Gradient clipping** — `max_norm=1.0` for stability
- ✅ **Feature extractor stored in model** — no global dependency

## Architecture
- **WavLM encoder** (FROZEN)
- **Attention pooling**
- **21-dim prosody** (pre-extracted)
- **Simple late fusion** → MLP classifier

## Expected: Val F1 ~0.75+

In [ ]:
# ============================================================
# SETUP
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/chuckle_checkpoints'
AUDIO_DIR = f'{BASE}/audio'
PROSODY_PATH = f'{BASE}/prosody_phaseD.json'
OUTPUT_DIR = f'{BASE}'
AUDIO_CKPT = f'{BASE}/phaseA_prosody_best.pt'
CHECKPOINT_DIR = f'{BASE}/checkpoints'

import os, json, time, math, numpy as np, librosa, torch
from sklearn.metrics import f1_score, precision_score, recall_score
import gc
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*mismatched key_padding_mask.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*attn_mask.*')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SR = 16000
MAX_DURATION = 5.0
MAX_LEN = int(MAX_DURATION * SR)
BATCH = 16
EPOCHS = 10
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('⚠️  WARNING: No GPU! Training will be very slow.')


In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================
HELD_OUT = {'BFIHCzw3itk', 'BAD4askmGgk', '1Nb3_os4RSA'}

print('Loading aligned_utterances...')
au_path = f'{BASE}/aligned_utterances.jsonl'
all_data = [json.loads(line) for line in open(au_path)]
print(f'  Total: {len(all_data)}')

held_out = [d for d in all_data if d['video_id'] in HELD_OUT]
in_dist = [d for d in all_data if d['video_id'] not in HELD_OUT]
print(f'  In-dist: {len(in_dist)}, Held-out test: {len(held_out)}')

np.random.seed(SEED)
np.random.shuffle(in_dist)
n_train = int(len(in_dist) * 0.85)
train, val = in_dist[:n_train], in_dist[n_train:]
test = held_out

print(f'  Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')
print(f'  Train pos%={100*sum(d["label_any"] for d in train)/len(train):.1f}%')
print(f'  Val pos%={100*sum(d["label_any"] for d in val)/len(val):.1f}%')
print(f'  Test pos%={100*sum(d["label_any"] for d in test)/len(test):.1f}%')

assert len(set(d['video_id'] for d in train) & set(d['video_id'] for d in held_out)) == 0, 'Leak!'
print('  ✅ No held-out leakage')


In [ ]:
# ============================================================
# LOAD PROSODY
# ============================================================
print('Loading prosody cache...')
with open(PROSODY_PATH) as f:
    prosody_raw = json.load(f)
prosody_map = {d['uid']: d['feats'] for d in prosody_raw}
print(f'  {len(prosody_map)} entries, dim={len(prosody_raw[0]["feats"])}')

from sklearn.preprocessing import StandardScaler
train_prosody = np.array([prosody_map.get(f"{d['video_id']}_{d['start']:.2f}", np.zeros(21)) for d in train])
prosody_scaler = StandardScaler()
prosody_scaler.fit(train_prosody)
print('  Prosody scaler fitted')

def get_prosody(d):
    key = f"{d['video_id']}_{d['start']:.2f}"
    feats = prosody_map.get(key, np.zeros(21))
    return prosody_scaler.transform([feats])[0]


In [ ]:
# ============================================================
# STREAMING AUDIO LOADER
# ============================================================
def load_audio_segment(d, sr=SR, max_duration=MAX_DURATION):
    vid = d['video_id']
    start = d['start']
    audio_path = f'{AUDIO_DIR}/{vid}.mp3'
    try:
        offset = max(0, start - 0.05)
        audio, _ = librosa.load(audio_path, sr=sr, mono=True, offset=offset, duration=max_duration)
        target_len = int(max_duration * sr)
        if len(audio) < target_len:
            audio = np.pad(audio, (0, target_len - len(audio)))
        return audio[:target_len]
    except Exception as e:
        print(f'  Warning: could not load {vid} at {start:.2f}s: {e}')
        target_len = int(max_duration * sr)
        return np.zeros(target_len, dtype=np.float32)

print('Testing streaming audio loader...')
_test = load_audio_segment(train[0])
print(f'  Shape: {_test.shape}, dtype: {_test.dtype}')
print('  ✅ Ready')


In [ ]:
# ============================================================
# MODEL
# ============================================================
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
import torch.nn as nn

print('Loading WavLM (FROZEN)...')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
wavlm.eval()
for p in wavlm.parameters():
    p.requires_grad = False
wavlm = wavlm.to(device)
print(f'  WavLM frozen: {sum(p.numel() for p in wavlm.parameters()):,} params')

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
print('  Feature extractor ready')

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, hidden_states):
        weights = self.attn(hidden_states).squeeze(-1)
        weights = torch.softmax(weights, dim=-1)
        pooled = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)
        return pooled, weights

class PhaseAWithProsody(nn.Module):
    def __init__(self, encoder, feat_extractor, prosody_dim=21, hidden=128):
        super().__init__()
        self.encoder = encoder
        self.feat_extractor = feat_extractor
        self.audio_pool = AttentionPooling(768)
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64, 32)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(800, hidden), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(hidden, 64), nn.GELU(), nn.Linear(64, 2)
        )
    def forward(self, waveforms, prosody_feats):
        wav_list = [w.cpu().numpy() for w in waveforms]
        inputs = self.feat_extractor(wav_list, sampling_rate=SR, return_tensors='pt', padding=False)
        inputs = {k: v.to(waveforms.device) for k, v in inputs.items()}
        with torch.no_grad():
            hidden = self.encoder(**inputs).last_hidden_state
        audio_emb, _ = self.audio_pool(hidden)
        prosody_emb = self.prosody_proj(prosody_feats)
        fused = torch.cat([audio_emb, prosody_emb], dim=1)
        return self.classifier(fused)

model = PhaseAWithProsody(wavlm, feature_extractor, prosody_dim=21, hidden=128).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {n_total:,} total | {n_trainable:,} trainable')


In [ ]:
# ============================================================
# OPTIMIZER
# ============================================================
classifier_params = (
    list(model.audio_pool.parameters()) +
    list(model.prosody_proj.parameters()) +
    list(model.classifier.parameters())
)
optimizer = torch.optim.AdamW(classifier_params, lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
class_weights = torch.tensor([1.0, 2.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print('Optimizer ready')


In [ ]:
# ============================================================
# CHECKPOINT SAVE / LOAD
# ============================================================
CHECKPOINT_PATH = f'{CHECKPOINT_DIR}/latest.pt'

def save_checkpoint(epoch, model, optimizer, scheduler, best_val_f1, history, best_state, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_f1': best_val_f1,
        'history': history,
        'best_state_dict': best_state if best_state is not None else {k: v.cpu().clone() for k, v in model.state_dict().items()},
    }, path)
    print(f'  💾 Checkpoint saved: {path}')

def load_checkpoint(model, optimizer, scheduler, path):
    if not os.path.exists(path):
        print(f'  No checkpoint at {path}, starting fresh')
        return 0, 0, None, {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f'  ✅ Resumed from epoch {ckpt["epoch"]+1}, best_val_f1={ckpt["best_val_f1"]:.4f}')
    best_state = {k: v.cpu().clone() for k, v in ckpt['best_state_dict'].items()} if 'best_state_dict' in ckpt else None
    return ckpt['epoch'] + 1, ckpt['best_val_f1'], best_state, ckpt['history']

print('Checkpoint functions ready')


In [ ]:
# ============================================================
# RESUME SETUP (set RESUME=True and run before training)
# ============================================================
RESUME = False  # ← Change to True to resume from checkpoint

if RESUME:
    start_epoch, best_val_f1, best_state, history = load_checkpoint(
        model, optimizer, scheduler, CHECKPOINT_PATH
    )
else:
    start_epoch = 0
    best_val_f1 = 0
    best_state = None
    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
    print('Starting from scratch')

print(f'  start_epoch={start_epoch}, best_val_f1={best_val_f1:.4f}')


In [ ]:
# ============================================================
# TRAINING & EVAL FUNCTIONS
# ============================================================
def train_epoch(model, data, optimizer, criterion, device, epoch, batch_size=BATCH):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = math.ceil(len(data) / batch_size)
    np.random.seed(SEED + epoch)
    indices = np.random.permutation(len(data))
    for i in range(0, len(data), batch_size):
        batch_idx = indices[i:i+batch_size]
        batch = [data[j] for j in batch_idx]
        waveforms, prosody_feats, labels = [], [], []
        for d in batch:
            audio = load_audio_segment(d)
            waveforms.append(torch.FloatTensor(audio).unsqueeze(0))
            prosody_feats.append(get_prosody(d))
            labels.append(d['label_any'])
        waveforms = torch.stack(waveforms).squeeze(1).to(device)
        prosody_feats = torch.FloatTensor(np.array(prosody_feats)).to(device)
        labels = torch.LongTensor(labels).to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(waveforms, prosody_feats)
        loss = criterion(logits, labels)
        if torch.isnan(loss) or torch.isinf(loss):
            print(f'  ⚠️ NaN/Inf loss at batch {i//batch_size}, skipping')
            del waveforms, prosody_feats, labels
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        del waveforms, prosody_feats, labels, logits, loss
        gc.collect()
    avg_loss = total_loss / n_batches
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    return avg_loss, f1

def eval_epoch(model, data, criterion, device, batch_size=BATCH):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = math.ceil(len(data) / batch_size)
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        waveforms, prosody_feats, labels = [], [], []
        for d in batch:
            audio = load_audio_segment(d)
            waveforms.append(torch.FloatTensor(audio).unsqueeze(0))
            prosody_feats.append(get_prosody(d))
            labels.append(d['label_any'])
        waveforms = torch.stack(waveforms).squeeze(1).to(device)
        prosody_feats = torch.FloatTensor(np.array(prosody_feats)).to(device)
        labels = torch.LongTensor(labels).to(device)
        with torch.no_grad():
            logits = model(waveforms, prosody_feats)
            loss = criterion(logits, labels)
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        del waveforms, prosody_feats, labels, logits, loss
        gc.collect()
    avg_loss = total_loss / n_batches
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    return avg_loss, f1, all_preds, all_labels

print('Training functions defined ✅')


In [ ]:
# ============================================================
# TRAIN (checkpoint every epoch)
# ============================================================
print('Starting training...')
print(f'  Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')
print(f'  Batch: {BATCH}, Epochs: {start_epoch+1}-{EPOCHS}')
print()

t0 = time.time()
for epoch in range(start_epoch, EPOCHS):
    epoch_t0 = time.time()
    train_loss, train_f1 = train_epoch(model, train, optimizer, criterion, device, epoch)
    val_loss, val_f1, _, _ = eval_epoch(model, val, criterion, device)
    scheduler.step()
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    improved = val_f1 > best_val_f1
    if improved:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    save_checkpoint(epoch, model, optimizer, scheduler, best_val_f1, history, best_state, CHECKPOINT_PATH)
    marker = ' ✅ NEW BEST' if improved else ''
    print(f'Epoch {epoch+1}/{EPOCHS} ({time.time()-epoch_t0:.0f}s, total {(time.time()-t0)/60:.1f}m){marker}')
    print(f'  Train loss={train_loss:.4f} f1={train_f1:.4f}')
    print(f'  Val   loss={val_loss:.4f} f1={val_f1:.4f}  (best={best_val_f1:.4f})')
    print()

if best_state is not None:
    torch.save(best_state, AUDIO_CKPT)
    print(f'Best model saved: {AUDIO_CKPT}')
print(f'Training complete! Best val F1: {best_val_f1:.4f}')
print(f'Total time: {(time.time()-t0)/60:.1f} minutes')


In [ ]:
# ============================================================
# EVALUATE ON HELD-OUT TEST
# ============================================================
if best_state is not None:
    model.load_state_dict(best_state)

print('Evaluating on held-out test set...')
test_loss, test_f1, test_preds, test_labels = eval_epoch(model, test, criterion, device)
print(f'  Test F1={test_f1:.4f}')
print(f'  Precision={precision_score(test_labels, test_preds, zero_division=0):.4f}')
print(f'  Recall={recall_score(test_labels, test_preds, zero_division=0):.4f}')

results = {
    'best_val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_precision': float(precision_score(test_labels, test_preds, zero_division=0)),
    'test_recall': float(recall_score(test_labels, test_preds, zero_division=0)),
    'history': {k: [float(x) for x in v] for k, v in history.items()},
}
with open(f'{OUTPUT_DIR}/phaseA_prosody_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved: {OUTPUT_DIR}/phaseA_prosody_results.json')

del best_state
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()
